
# Tool Wear State Classification (Dense NN, One-Hot Encoding)

This notebook demonstrates a very simple **multi-class classification** for manufacturing:
- **Inputs (process parameters):** `spindle_speed`, `feed`, `doc`
- **Output (target):** `wear_state` with classes `0=new`, `1=slightly worn`, `2=moderately worn`, `3=critical`

**Assumptions (constants):** `material`, `type_of_operation`, and `time_machined` are treated as constants and not included as inputs.

> The code is intentionally **simple** and uses common libraries similar to a fundamental ANN class.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import utils  # for to_categorical

# Optional: easy local file upload in Colab
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


## Load locally stored Excel data

In [ ]:

# If running in Colab, either place the file in /content or upload it when prompted.
DATA_PATH = "tool_wear_dataset.xlsx"  # expected name in the working directory

import os

if not os.path.exists(DATA_PATH):
    if IN_COLAB:
        print("Select the local Excel file (tool_wear_dataset.xlsx) to upload...")
        _uploaded = files.upload()  # pick your local file
        # If the uploaded file has a different name, adjust DATA_PATH accordingly.
        if DATA_PATH not in _uploaded:
            # just pick the first uploaded file name
            DATA_PATH = list(_uploaded.keys())[0]
    else:
        raise FileNotFoundError(f"Could not find {DATA_PATH}. Please place it next to this notebook.")

data = pd.read_excel(DATA_PATH)
print("Data shape:", data.shape)
data.head()


## Train/test split (80/20), normalize inputs, one-hot encode target

In [ ]:

# Shuffle indices and split 80/20 without sklearn, to keep things simple
rng = np.random.default_rng(0)
indices = np.arange(len(data))
rng.shuffle(indices)

split = int(0.8 * len(indices))
train_idx, test_idx = indices[:split], indices[split:]

# Separate features and target
X = data[["spindle_speed", "feed", "doc"]].values.astype(np.float32)
y = data["wear_state"].values.astype(np.int32)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Min-max normalize using TRAIN statistics
xmin = X_train.min(axis=0, keepdims=True)
xmax = X_train.max(axis=0, keepdims=True)
X_train_norm = (X_train - xmin) / (xmax - xmin + 1e-8)
X_test_norm  = (X_test  - xmin) / (xmax - xmin + 1e-8)

# One-hot encode the wear_state (4 classes)
NUM_CLASSES = 4
y_train_oh = utils.to_categorical(y_train, num_classes=NUM_CLASSES)
y_test_oh  = utils.to_categorical(y_test, num_classes=NUM_CLASSES)

print("X_train_norm shape:", X_train_norm.shape)
print("y_train_oh shape:", y_train_oh.shape)


## Build a simple Dense Neural Network

In [ ]:

model = Sequential()
model.add(Dense(16, activation="relu", input_shape=(3,)))  # 3 inputs: spindle_speed, feed, doc
model.add(Dense(16, activation="relu"))
model.add(Dense(4, activation="softmax"))  # 4 wear-state classes

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()


## Train the model

In [ ]:

history = model.fit(
    X_train_norm, y_train_oh,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0  # set to 1 to see per-epoch logs
)
print("Training complete.")


## Plot training curves

In [ ]:

plt.figure()
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy")

plt.figure()
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")


## Evaluate on the test set

In [ ]:

test_loss, test_acc = model.evaluate(X_test_norm, y_test_oh, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")


## Sample predictions vs. ground truth

In [ ]:

# Show a few predictions vs labels
idx = np.arange(len(X_test_norm))
rng.shuffle(idx)
idx = idx[:10]

y_prob = model.predict(X_test_norm[idx], verbose=0)
y_pred = y_prob.argmax(axis=1)

print("Pred (y)  | True (y)")
for yp, yt in zip(y_pred, y_test[idx]):
    print(f"{int(yp):>9} | {int(yt):>8}")
